In [14]:
import cv2
import sys
import glob
import numpy as np
import pandas as pd 
import matplotlib
import matplotlib.pyplot as plt
sys.path.append(r"/Users/johnmarshall/Documents/Analysis/post_cnmfe_analysis")
import dlc_utils as dlc
import os
import datetime
import math
import time 
from tqdm import tqdm
from matplotlib.dates import DateFormatter, date2num
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:98% !important; }</style>"))

/var/folders/fc/24x3k2m92bvbv7ck1v5mt3n40000gn/T/ipykernel_16879/3730779655.py:16: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython display
  from IPython.core.display import display, HTML


In [15]:
#save path
base_dir = r'/Users/johnmarshall/Documents/Analysis/miniscope_analysis/SCI_analysis/velocityAlignmentAnalysis/IntroFigure/DIO_r2.25_11_28_12/'
savePath = base_dir 

#params 
behavCamFrameRate=15
miniscopeCamFrameRate=20
downSampleFactorForVelocity = '.2S'
medianFilter = '1S'
sessionPath = savePath+'DIO_r2.25_11_28_12_combined_LocationOutput.csv'

print(sessionPath.split(os.sep)[:-1])
# load eZ track output and behavior camera timestamps from miniscope software 
ezTrackOutput = pd.read_csv(sessionPath)
timestampfile = pd.read_table('/'.join(sessionPath.split(os.sep)[:-1])+'/timeStamps.csv', delimiter=',')
miniscope_timestampfile = pd.read_table('/'.join(sessionPath.split(os.sep)[:-1])+'/timeStampsMiniscope.csv', delimiter=',')
    
timestampfile_td = timestampfile.set_index(pd.to_timedelta(np.linspace(0, (len(timestampfile)-1)*(1/behavCamFrameRate), len(timestampfile)), unit='s'), drop=False)
  
miniscopetimestamp_td_originalHz = miniscope_timestampfile.set_index(pd.to_timedelta(np.linspace(0, (len(miniscope_timestampfile)-1)*(1/miniscopeCamFrameRate), len(miniscope_timestampfile)), unit='s'), drop=False)

['', 'Users', 'johnmarshall', 'Documents', 'Analysis', 'miniscope_analysis', 'SCI_analysis', 'velocityAlignmentAnalysis', 'IntroFigure', 'DIO_r2.25_11_28_12']


In [16]:
timestampfile_td.head()

,Frame Number,Time Stamp (ms),Buffer Index
0 days 00:00:00,0,-63,0
0 days 00:00:00.066666667,1,18,0
0 days 00:00:00.133333333,2,83,0
0 days 00:00:00.200000,3,146,0
0 days 00:00:00.266666667,4,209,0


In [17]:
miniscopetimestamp_td_originalHz.head()

,Frame Number,Time Stamp (ms),Buffer Index
0 days 00:00:00,0,-63,0
0 days 00:00:00.050000,1,18,0
0 days 00:00:00.100000,2,83,0
0 days 00:00:00.150000,3,146,0
0 days 00:00:00.200000,4,209,0


In [18]:
behavCam_frames = []
sys_clock_behavCam = []
#create "key" for aligning miniscope frames to timestamp file
#then create behavior TD and align
for msCam_frame in tqdm(range(0, len(miniscopetimestamp_td_originalHz['Frame Number']))):
    #get sys clock time of each miniscope recorded frame
    #sys_clock_msCam = time_stamps['sysClock'].loc[msCam_frame]
    #find behav cam frame closest to sys clock time of ms frame
    behavCam_frame = list(timestampfile_td.iloc[(timestampfile_td['Time Stamp (ms)']-miniscopetimestamp_td_originalHz['Time Stamp (ms)'].iloc[msCam_frame]).abs().argsort()[:1]].index)[0]
    #this is the behavCamIndex that is closest to the corresponding miniscope frame 
    behavCam_frames.append(behavCam_frame)
    sys_clock_behavCam.append(timestampfile_td.loc[behavCam_frame]['Time Stamp (ms)'])

behavCamIdxToAlign = [timestampfile_td.index.get_loc(idx) for idx in behavCam_frames]
#ezTrackOutput
miniscopetimestamp_td_originalHz['closestBehavCamFrameIdx'] = behavCamIdxToAlign

X_coor=[]
Y_coor=[]
Distance_px=[] 
Distance_cm=[]

for i in tqdm(miniscopetimestamp_td_originalHz['closestBehavCamFrameIdx'].values):
    X_coor.append(ezTrackOutput.loc[i]['X'])
    Y_coor.append(ezTrackOutput.loc[i]['Y'])
    Distance_px.append(ezTrackOutput.loc[i]['Distance_px'])
    Distance_cm.append(ezTrackOutput.loc[i]['Distance_cm'])
    
miniscopetimestamp_td_originalHz['X_coor'] = X_coor
miniscopetimestamp_td_originalHz['Y_coor'] = Y_coor
miniscopetimestamp_td_originalHz['Distance_px'] = Distance_px
miniscopetimestamp_td_originalHz['Distance_cm'] = Distance_cm

100%|██████████████████████████████████| 22883/22883 [00:01<00:00, 14750.53it/s]


In [19]:
# calculate cmPerPixel
cmPerPixel = ezTrackOutput['Distance_cm'][1]/ezTrackOutput['Distance_px'][1]

print(cmPerPixel)
def euclideanDistance(row):
    return(math.sqrt(row['X_coor']**2+row['Y_coor']**2))  
def euclideanDistancePixels(row, cmPerPixel):
    return(math.sqrt(row['X_coor']**2+row['Y_coor']**2)*cmPerPixel)
def calcVelocity(row, cmPerPixel, samplingRate):
    #euclidean distance divided by sampling rate, multiplied by cm/pixel
    return((math.sqrt(row['X_coor']**2+row['Y_coor']**2)/samplingRate)*cmPerPixel)
    
# 
samplingRateOrig = timestampfile['Time Stamp (ms)'].diff().mean()/1000
print('orig sampling rate')
print(samplingRateOrig)
miniscopetimestamp_td_originalHz['Velocity'] = miniscopetimestamp_td_originalHz.diff().apply(lambda row : calcVelocity(row, cmPerPixel, samplingRateOrig), axis=1)
miniscopetimestamp_td_originalHz['Distance'] = miniscopetimestamp_td_originalHz.diff().apply(lambda row : euclideanDistancePixels(row, cmPerPixel), axis=1)

## in doing the downsampling, break apart X_coor, Y_coor columns to to .mean() instead of .bfill()

# downsample to .1S
miniscopeTimeStampsDownsampled_1 = miniscopetimestamp_td_originalHz.resample('.1S').bfill()
X_Y_Downsampled_1 = miniscopetimestamp_td_originalHz[['X_coor', 'Y_coor']].resample('.1S').mean()
miniscopeTimeStampsDownsampled_1[['X_coor', 'Y_coor']] = X_Y_Downsampled_1

# downsample to .2S
miniscopeTimeStampsDownsampled_2 = miniscopetimestamp_td_originalHz.resample('.2S').bfill()
X_Y_Downsampled_2 = miniscopetimestamp_td_originalHz[['X_coor', 'Y_coor']].resample('.2S').mean()
miniscopeTimeStampsDownsampled_2[['X_coor', 'Y_coor']] = X_Y_Downsampled_2

#
samplingRate_1 = 0.1
miniscopeTimeStampsDownsampled_1['Velocity'] = miniscopeTimeStampsDownsampled_1.diff().apply(lambda row : calcVelocity(row, cmPerPixel, samplingRate_1), axis=1)
miniscopeTimeStampsDownsampled_1['Distance'] = miniscopeTimeStampsDownsampled_1.diff().apply(lambda row : euclideanDistancePixels(row, cmPerPixel), axis=1)

#
samplingRate_2 = 0.2
miniscopeTimeStampsDownsampled_2['Velocity'] = miniscopeTimeStampsDownsampled_2.diff().apply(lambda row : calcVelocity(row, cmPerPixel, samplingRate_2), axis=1)
miniscopeTimeStampsDownsampled_2['Distance'] = miniscopeTimeStampsDownsampled_2.diff().apply(lambda row : euclideanDistancePixels(row, cmPerPixel), axis=1)

# save these for comparison
miniscopetimestamp_td_originalHz.to_csv(savePath+os.path.split(sessionPath)[0].split('/')[-1]+'velocity_origHz_coor.csv')
miniscopeTimeStampsDownsampled_1.to_csv(savePath+os.path.split(sessionPath)[0].split('/')[-1]+'velocity_resampled_1_coor.csv')

miniscopeTimeStampsDownsampled_2.to_csv(savePath+os.path.split(sessionPath)[0].split('/')[-1]+'velocity_resampled_2_coor.csv')

0.11474790445137907
orig sampling rate
0.06666829822567957


/var/folders/fc/24x3k2m92bvbv7ck1v5mt3n40000gn/T/ipykernel_16879/2389678762.py:23: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  miniscopeTimeStampsDownsampled_1 = miniscopetimestamp_td_originalHz.resample('.1S').bfill()
/var/folders/fc/24x3k2m92bvbv7ck1v5mt3n40000gn/T/ipykernel_16879/2389678762.py:24: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  X_Y_Downsampled_1 = miniscopetimestamp_td_originalHz[['X_coor', 'Y_coor']].resample('.1S').mean()
/var/folders/fc/24x3k2m92bvbv7ck1v5mt3n40000gn/T/ipykernel_16879/2389678762.py:28: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  miniscopeTimeStampsDownsampled_2 = miniscopetimestamp_td_originalHz.resample('.2S').bfill()
/var/folders/fc/24x3k2m92bvbv7ck1v5mt3n40000gn/T/ipykernel_16879/2389678762.py:29: FutureWarning: 'S' is deprecated and will be removed in a future version, please use '

In [20]:
#ezTrackOutput

In [21]:
miniscopetimestamp_td_originalHz

,Frame Number,Time Stamp (ms),Buffer Index,closestBehavCamFrameIdx,X_coor,Y_coor,Distance_px,Distance_cm,Velocity,Distance
0 days 00:00:00,0,-63,0,0,77.478320,321.671022,0.000000,0.000000,NaN,NaN
0 days 00:00:00.050000,1,18,0,1,75.824801,323.341351,2.350346,0.269697,4.045359,0.269697
0 days 00:00:00.100000,2,83,0,2,74.032453,323.623327,1.814393,0.208198,3.122890,0.208198
0 days 00:00:00.150000,3,146,0,3,72.684014,322.579388,1.705314,0.195681,2.935147,0.195681
0 days 00:00:00.200000,4,209,0,4,72.754347,322.310042,0.278378,0.031943,0.479137,0.031943
...,...,...,...,...,...,...,...,...,...,...
0 days 00:19:03.900000,22878,1525172,0,22878,187.201022,258.597554,2.424255,0.278178,4.172570,0.278178
0 days 00:19:03.950000,22879,1525232,0,22879,185.801947,258.661881,1.400552,0.160710,2.410597,0.160710
0 days 00:19:04,22880,1525313,0,22880,185.712860,258.960705,0.311820,0.035781,0.536698,0.035781
0 days 00:19:04.050000,22881,1525376,0,22881,186.424870,259.194548,0.749427,0.085995,1.289896,0.085995


In [22]:
miniscopeTimeStampsDownsampled_2[0:20]

,Frame Number,Time Stamp (ms),Buffer Index,closestBehavCamFrameIdx,X_coor,Y_coor,Distance_px,Distance_cm,Velocity,Distance
0 days 00:00:00,0,-63,0,0,75.004897,322.803772,0.000000,0.000000,NaN,NaN
0 days 00:00:00.200000,4,209,0,4,73.256716,319.696780,0.278378,0.031943,2.045406,0.409081
0 days 00:00:00.400000,8,483,0,8,75.915683,315.081938,0.202318,0.023216,3.055769,0.611154
0 days 00:00:00.600000,12,738,0,12,80.173025,307.404210,8.437244,0.968156,5.036913,1.007383
0 days 00:00:00.800000,16,1010,0,16,74.187741,309.946065,0.483641,0.055497,3.730836,0.746167
0 days 00:00:01,20,1286,0,20,73.504525,308.801468,1.830338,0.210027,0.764794,0.152959
0 days 00:00:01.200000,24,1537,0,24,69.954660,311.934861,2.167296,0.248693,2.716624,0.543325
0 days 00:00:01.400000,28,1810,0,28,71.044158,313.031719,2.024367,0.232292,0.886999,0.177400
0 days 00:00:01.600000,32,2084,0,32,76.938074,310.134056,1.298000,0.148943,3.768150,0.753630
0 days 00:00:01.800000,36,2338,0,36,76.450262,309.062918,5.650168,0.648345,0.675284,0.135057


In [23]:
## check alignment with video overlay 
alignedTracesLocationData = miniscopetimestamp_td_originalHz[['closestBehavCamFrameIdx', 'X_coor', 'Y_coor', 'Velocity']]
locationReIndexedToBehaviorData = (
    alignedTracesLocationData
      .groupby('closestBehavCamFrameIdx')           # group by frame-idx
      .mean(numeric_only=True)                     # mean of only number-dtype cols
)

In [24]:
# ── CONFIG ─────────────────────────────────────────────────────────────────────

dirPath      = savePath
rotatedVideo = 'video_output.avi'
input_vid    = os.path.join(dirPath, rotatedVideo)
output_vid   = os.path.join(dirPath, 'with_velocity_overlay.mp4')

vel_array = locationReIndexedToBehaviorData['Velocity'].to_numpy()
x_pos     = locationReIndexedToBehaviorData['X_coor'].to_numpy()
y_pos     = locationReIndexedToBehaviorData['Y_coor'].to_numpy()

bar_h    = 50  # height of black bar

# ── SET UP I/O ─────────────────────────────────────────────────────────────────

cap      = cv2.VideoCapture(input_vid)
fps      = cap.get(cv2.CAP_PROP_FPS)
w        = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h        = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

fourcc = cv2.VideoWriter_fourcc(*'XVID')
out    = cv2.VideoWriter(output_vid, fourcc, fps, (w, h + bar_h))

# text & dot styling
font        = cv2.FONT_HERSHEY_SIMPLEX
font_scale  = 1
thickness   = 2
margin      = 10
dot_radius  = 5

# ── PROCESS FRAMES ──────────────────────────────────────────────────────────────

for frame_idx in tqdm(range(len(vel_array)), desc="Adding overlay"):
    ret, frame = cap.read()
    if not ret:
        break

    # 1) overlay the dot on the ORIGINAL frame at (x_pos, y_pos)
    #    clamp coordinates to [0, w-1] and [0, h-1]
    xm = int(np.clip(x_pos[frame_idx], 0, w - 1))
    ym = int(np.clip(y_pos[frame_idx], 0, h - 1))
    cv2.circle(frame, (xm, ym), dot_radius, (255, 255, 255), -1)

    # 2) create the black bar underneath, then stack
    bar       = np.zeros((bar_h, w, 3), dtype=np.uint8)
    new_frame = np.vstack((frame, bar))

    # 3) overlay the velocity text in the bar (left‐aligned)
    vel_text = f"{vel_array[frame_idx]:.2f} cm/s"
    (tw, th_text), _ = cv2.getTextSize(vel_text, font, font_scale, thickness)
    text_x = margin
    text_y = h + margin + th_text
    cv2.putText(
        new_frame,
        vel_text,
        (text_x, text_y),
        font,
        font_scale,
        (255, 255, 255),
        thickness,
        cv2.LINE_AA
    )

    out.write(new_frame)

# ── CLEAN UP ────────────────────────────────────────────────────────────────────

cap.release()
out.release()
print("Done – saved with overlay to:", output_vid)

OpenCV: FFMPEG: tag 0x44495658/'XVID' is not supported with codec id 12 and format 'mp4 / MP4 (MPEG-4 Part 14)'
OpenCV: FFMPEG: fallback to use tag 0x7634706d/'mp4v'
Adding overlay: 100%|███████████████████| 22883/22883 [00:18<00:00, 1238.98it/s]

Done – saved with overlay to: /Users/johnmarshall/Documents/Analysis/miniscope_analysis/SCI_analysis/velocityAlignmentAnalysis/IntroFigure/DIO_r2.25_11_28_12/with_velocity_overlay.mp4
